# 0. Title

# Multivariate marked Hawkes–GPD model for extreme loss clustering and spillover in Vietnamese equities

**Notebook type:** Academic research template  
**Workflow principle:** start from transparent benchmark models and escalate complexity only when diagnostics indicate unmet assumptions or inadequate tail-risk performance.  

**Model ladder:** Historical VaR/ES → Poisson–GPD → Hawkes–GPD → marked Hawkes–GPD → multivariate marked Hawkes–GPD.


## 1. Abstract

This notebook provides a reproducible research template for studying clustering and cross-market spillover of extreme equity losses in Vietnamese equities. The empirical strategy begins with non-parametric Historical VaR/ES, then progressively introduces peaks-over-threshold tail modelling, self-exciting event arrivals, loss-size marks, and multivariate excitation across assets. Each modelling stage includes diagnostics, and movement to a more complex model is justified only when the current model fails to adequately describe exceedance frequency, tail severity, temporal clustering, or spillover dependence.


## 2. Research question

**Main question.** Do extreme losses in Vietnamese equities exhibit statistically and economically meaningful temporal clustering and cross-asset spillover that require a multivariate marked Hawkes–GPD specification rather than simpler VaR/ES, Poisson–GPD, or univariate Hawkes–GPD models?

**Sub-questions.**

1. Are unconditional Historical VaR and ES adequate for backtesting extreme downside risk?
2. Are threshold exceedances compatible with an independent Poisson arrival process?
3. Does a self-exciting Hawkes process improve the modelling of clustered exceedance times?
4. Do exceedance magnitudes contain mark information that alters future arrival intensity or tail risk?
5. Are there significant cross-asset excitation channels among Vietnamese equities?


## 3. Data description

**Suggested data unit:** daily adjusted close prices for Vietnamese equities or indices, for example VN30 constituents, sector indices, or liquid HoSE/HNX stocks.

**Required fields.**

| Column | Description |
|---|---|
| `date` | Trading date |
| `asset` | Ticker or asset identifier |
| `adj_close` | Adjusted closing price |
| Optional controls | Volume, sector, market index, macro variables |

**Constructed variables.**

- Log return: $r_{i,t}=\log(P_{i,t})-\log(P_{i,t-1})$.
- Loss: $L_{i,t}=-r_{i,t}$.
- Extreme-loss event: $N_{i,t}=1\{L_{i,t}>u_i\}$ for threshold $u_i$.
- Mark: exceedance severity $Y_{i,t}=L_{i,t}-u_i$ conditional on $L_{i,t}>u_i$.


In [ ]:
# 3.1 Environment and configuration
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:  # Notebook users should install matplotlib for figures.
    plt = None

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = Path("data/vietnam_equities.csv")  # Replace with the project data file.
DATE_COL = "date"
ASSET_COL = "asset"
PRICE_COL = "adj_close"
TAIL_PROB = 0.01
THRESHOLD_Q = 0.95


In [ ]:
# 3.2 Data loader with a small synthetic fallback for template execution

def load_equity_data(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path, parse_dates=[DATE_COL])
        required = {DATE_COL, ASSET_COL, PRICE_COL}
        missing = required.difference(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")
        return df.sort_values([ASSET_COL, DATE_COL]).reset_index(drop=True)

    # Synthetic fallback keeps the notebook runnable before proprietary data are added.
    dates = pd.bdate_range("2018-01-01", periods=900)
    assets = ["AAA", "BBB", "CCC"]
    rows = []
    for j, asset in enumerate(assets):
        shocks = np.random.standard_t(df=5, size=len(dates)) * (0.012 + 0.002 * j)
        prices = 100 * np.exp(np.cumsum(shocks))
        rows.extend({DATE_COL: d, ASSET_COL: asset, PRICE_COL: p} for d, p in zip(dates, prices))
    return pd.DataFrame(rows)

prices = load_equity_data(DATA_PATH)
prices.head()


In [ ]:
# 3.3 Return, loss, threshold, event, and mark construction
panel = prices.copy()
panel["log_price"] = np.log(panel[PRICE_COL])
panel["return"] = panel.groupby(ASSET_COL)["log_price"].diff()
panel["loss"] = -panel["return"]
panel = panel.dropna(subset=["loss"]).reset_index(drop=True)

thresholds = panel.groupby(ASSET_COL)["loss"].quantile(THRESHOLD_Q).rename("threshold")
panel = panel.join(thresholds, on=ASSET_COL)
panel["event"] = (panel["loss"] > panel["threshold"]).astype(int)
panel["mark"] = np.where(panel["event"].eq(1), panel["loss"] - panel["threshold"], np.nan)

panel.groupby(ASSET_COL).agg(
    start=(DATE_COL, "min"),
    end=(DATE_COL, "max"),
    observations=("loss", "size"),
    threshold=("threshold", "first"),
    exceedances=("event", "sum"),
)


## 4. Methodology

The notebook uses a **sequential escalation rule**.

1. Fit the simplest model that addresses the research need.
2. Diagnose whether core assumptions are plausible.
3. Escalate only if diagnostics reveal a substantive deficiency.
4. Record the reason for escalation in the comparison table.

### 4.1 Escalation diagnostics

| Current model | Diagnostic trigger for escalation |
|---|---|
| Historical VaR/ES | VaR exception rate or ES exceedance severity is unstable, clustered, or fails backtesting |
| Poisson–GPD | Exceedance inter-arrival times are not memoryless; event counts are overdispersed; residual clustering remains |
| Hawkes–GPD | Event timing improves, but exceedance sizes predict future intensity or risk |
| Marked Hawkes–GPD | Asset-specific dynamics remain connected by simultaneous or lagged spillovers |
| Multivariate marked Hawkes–GPD | Final model; diagnose calibration, stability, and spillover interpretability |


In [ ]:
# 4.2 Shared diagnostic helpers

def historical_var_es(losses: pd.Series, alpha: float = TAIL_PROB) -> tuple[float, float]:
    var = losses.quantile(1 - alpha)
    es = losses.loc[losses >= var].mean()
    return float(var), float(es)


def exception_summary(losses: pd.Series, var: float) -> dict:
    exceptions = losses > var
    rate = exceptions.mean()
    clusters = int(((exceptions.astype(int).diff() == 0) & exceptions).sum())
    return {"exceptions": int(exceptions.sum()), "exception_rate": float(rate), "adjacent_exception_count": clusters}


def dispersion_index(events: pd.Series) -> float:
    counts = events.groupby(events.index // 21).sum()  # approximate monthly blocks for daily data
    return float(counts.var(ddof=1) / counts.mean()) if counts.mean() > 0 else np.nan


def should_escalate(metrics: dict, reasons: list[str]) -> bool:
    print("Diagnostics:", metrics)
    if reasons:
        print("Escalation reasons:")
        for reason in reasons:
            print(f"- {reason}")
        return True
    print("No clear escalation trigger at this stage.")
    return False


## 5. Empirical results: Model 1 — Historical VaR/ES

Historical VaR/ES is the transparent benchmark. It does not impose a parametric tail distribution or event-arrival process, but it also cannot explain temporal clustering or spillover.


In [ ]:
# 5.1 Fit Historical VaR/ES by asset
hist_rows = []
for asset, g in panel.groupby(ASSET_COL):
    var, es = historical_var_es(g["loss"], TAIL_PROB)
    hist_rows.append({ASSET_COL: asset, "VaR": var, "ES": es, **exception_summary(g["loss"], var)})
historical_results = pd.DataFrame(hist_rows)
historical_results


## 6. Model diagnostics: Historical VaR/ES

Check whether exception rates are close to the target level and whether exceptions are isolated. A high adjacent-exception count indicates clustering not captured by unconditional Historical VaR/ES.


In [ ]:
# 6.1 Historical VaR/ES escalation decision
hist_metrics = historical_results.set_index(ASSET_COL).to_dict("index")
hist_reasons = []
if (historical_results["adjacent_exception_count"] > 0).any():
    hist_reasons.append("VaR exceptions occur in adjacent periods, suggesting temporal clustering.")
if (historical_results["exception_rate"] - TAIL_PROB).abs().max() > TAIL_PROB:
    hist_reasons.append("Exception rates materially deviate from the target tail probability.")
ESCALATE_TO_POISSON_GPD = should_escalate({"historical": hist_metrics}, hist_reasons)


## 7. Empirical results: Model 2 — Poisson–GPD

The Poisson–GPD model separates event frequency and exceedance severity. Exceedance arrivals are assumed independent with constant intensity, while exceedance magnitudes are modelled using a generalized Pareto distribution (GPD).

\[
N_i(t) \sim \text{Poisson}(\lambda_i t), \qquad Y_i=L_i-u_i \mid L_i>u_i \sim \text{GPD}(\xi_i,\beta_i).
\]


In [ ]:
# 7.1 Lightweight Poisson-GPD fit scaffold
try:
    from scipy.stats import genpareto
except ImportError:
    genpareto = None

poisson_gpd_rows = []
for asset, g in panel.groupby(ASSET_COL):
    event_count = int(g["event"].sum())
    intensity = event_count / len(g)
    marks = g.loc[g["event"].eq(1), "mark"].dropna()
    if genpareto is not None and len(marks) >= 10:
        xi, loc, beta = genpareto.fit(marks, floc=0)
    else:
        xi, beta = np.nan, np.nan
    poisson_gpd_rows.append({ASSET_COL: asset, "lambda": intensity, "gpd_xi": xi, "gpd_beta": beta, "n_exceed": event_count})
poisson_gpd_results = pd.DataFrame(poisson_gpd_rows)
poisson_gpd_results


## 8. Model diagnostics: Poisson–GPD

A Poisson process implies approximately equidispersed event counts and memoryless inter-arrival times. Overdispersion or clustered arrivals motivate a Hawkes arrival process.


In [ ]:
# 8.1 Poisson-GPD escalation decision
poisson_diag_rows = []
for asset, g in panel.groupby(ASSET_COL):
    ordered = g.sort_values(DATE_COL).reset_index(drop=True)
    di = dispersion_index(ordered["event"])
    event_idx = np.flatnonzero(ordered["event"].to_numpy())
    interarrival_cv = np.nan
    if len(event_idx) > 2:
        gaps = np.diff(event_idx)
        interarrival_cv = float(np.std(gaps, ddof=1) / np.mean(gaps))
    poisson_diag_rows.append({ASSET_COL: asset, "dispersion_index": di, "interarrival_cv": interarrival_cv})
poisson_diagnostics = pd.DataFrame(poisson_diag_rows)

pois_reasons = []
if (poisson_diagnostics["dispersion_index"] > 1.5).any():
    pois_reasons.append("Exceedance counts are overdispersed relative to a constant-intensity Poisson process.")
if (poisson_diagnostics["interarrival_cv"] > 1.25).any():
    pois_reasons.append("Inter-arrival times are more variable than expected under memoryless arrivals.")
ESCALATE_TO_HAWKES_GPD = should_escalate({"poisson_gpd": poisson_diagnostics.to_dict("records")}, pois_reasons)
poisson_diagnostics


## 9. Empirical results: Model 3 — Hawkes–GPD

The Hawkes–GPD model keeps the GPD tail for severities but replaces constant Poisson intensity with self-exciting intensity:

\[
\lambda_i(t)=\mu_i+\sum_{t_{i,k}<t}\alpha_i e^{-\beta_i(t-t_{i,k})}.
\]

This section intentionally provides a fitting scaffold rather than a full production optimizer. In an empirical paper, parameters should be estimated by maximum likelihood with positivity and stationarity constraints.


In [ ]:
# 9.1 Hawkes-GPD scaffold: simple moment-style placeholders for sequential workflow
hawkes_rows = []
for asset, g in panel.groupby(ASSET_COL):
    ordered = g.sort_values(DATE_COL).reset_index(drop=True)
    base_rate = ordered["event"].mean()
    adjacent = ((ordered["event"].eq(1)) & (ordered["event"].shift(1).eq(1))).sum()
    excitation_proxy = adjacent / max(int(ordered["event"].sum()), 1)
    hawkes_rows.append({
        ASSET_COL: asset,
        "mu_proxy": max(base_rate * (1 - excitation_proxy), 1e-8),
        "alpha_proxy": excitation_proxy,
        "beta_proxy": 1.0,
        "branching_ratio_proxy": excitation_proxy,
    })
hawkes_results = pd.DataFrame(hawkes_rows)
hawkes_results


## 10. Model diagnostics: Hawkes–GPD

For a fitted Hawkes process, transformed event times should resemble a unit-rate Poisson process. Additional checks include stationarity (branching ratio below one), residual clustering, and tail fit of GPD exceedances.


In [ ]:
# 10.1 Hawkes-GPD escalation decision
hawkes_reasons = []
if (hawkes_results["branching_ratio_proxy"] >= 0.8).any():
    hawkes_reasons.append("Estimated self-excitation is high, requiring careful marked or multivariate structure checks.")

mark_predictability = []
for asset, g in panel.groupby(ASSET_COL):
    ordered = g.sort_values(DATE_COL).reset_index(drop=True)
    ordered["lag_mark"] = ordered["mark"].fillna(0).shift(1).fillna(0)
    if ordered["lag_mark"].std() > 0:
        corr = ordered[["lag_mark", "event"]].corr().iloc[0, 1]
    else:
        corr = np.nan
    mark_predictability.append({ASSET_COL: asset, "corr_lag_mark_next_event": corr})
mark_diag = pd.DataFrame(mark_predictability)
if mark_diag["corr_lag_mark_next_event"].abs().fillna(0).max() > 0.05:
    hawkes_reasons.append("Lagged exceedance magnitudes show predictive association with future event arrivals.")
ESCALATE_TO_MARKED_HAWKES_GPD = should_escalate({"hawkes": hawkes_results.to_dict("records"), "mark_diag": mark_diag.to_dict("records")}, hawkes_reasons)
mark_diag


## 11. Empirical results: Model 4 — marked Hawkes–GPD

The marked Hawkes–GPD model allows exceedance magnitude to affect subsequent intensity:

\[
\lambda_i(t)=\mu_i+\sum_{t_{i,k}<t}\alpha_i g(Y_{i,k}) e^{-\beta_i(t-t_{i,k})},
\]

where $g(Y)$ can be linear, logarithmic, or otherwise constrained to preserve positivity.


In [ ]:
# 11.1 Marked Hawkes-GPD scaffold
marked_rows = []
for asset, g in panel.groupby(ASSET_COL):
    ordered = g.sort_values(DATE_COL).reset_index(drop=True)
    lag_mark = ordered["mark"].fillna(0).shift(1).fillna(0)
    mark_effect_proxy = np.corrcoef(lag_mark, ordered["event"])[0, 1] if lag_mark.std() > 0 else np.nan
    marked_rows.append({
        ASSET_COL: asset,
        "mu_proxy": ordered["event"].mean(),
        "alpha_proxy": hawkes_results.loc[hawkes_results[ASSET_COL].eq(asset), "alpha_proxy"].iloc[0],
        "mark_effect_proxy": mark_effect_proxy,
    })
marked_hawkes_results = pd.DataFrame(marked_rows)
marked_hawkes_results


## 12. Model diagnostics: marked Hawkes–GPD

After including marks, remaining dependence across assets should be investigated. Significant contemporaneous or lagged cross-asset event association motivates a multivariate marked Hawkes–GPD model.


In [ ]:
# 12.1 Marked Hawkes-GPD escalation decision using cross-asset event association
wide_events = panel.pivot_table(index=DATE_COL, columns=ASSET_COL, values="event", aggfunc="max").fillna(0)
lag_cross_corr = wide_events.shift(1).corrwith(wide_events, axis=0)
contemp_corr = wide_events.corr()
max_offdiag_corr = contemp_corr.where(~np.eye(len(contemp_corr), dtype=bool)).abs().max().max()

marked_reasons = []
if pd.notna(max_offdiag_corr) and max_offdiag_corr > 0.05:
    marked_reasons.append("Cross-asset extreme-loss events are correlated, indicating potential spillover.")
ESCALATE_TO_MULTIVARIATE_MARKED_HAWKES_GPD = should_escalate(
    {"max_offdiag_event_corr": float(max_offdiag_corr) if pd.notna(max_offdiag_corr) else np.nan},
    marked_reasons,
)
contemp_corr


## 13. Empirical results: Model 5 — multivariate marked Hawkes–GPD

The final model allows self-excitation and cross-excitation among assets:

\[
\lambda_i(t)=\mu_i+\sum_j\sum_{t_{j,k}<t}\alpha_{ij} g_j(Y_{j,k}) e^{-\beta_{ij}(t-t_{j,k})}.
\]

The matrix $A=(\alpha_{ij})$ summarizes spillover channels. Diagonal terms represent own-asset clustering; off-diagonal terms represent spillover from asset $j$ to asset $i$.


In [ ]:
# 13.1 Multivariate marked Hawkes-GPD scaffold: spillover proxy matrix
assets = list(wide_events.columns)
spillover_proxy = pd.DataFrame(index=assets, columns=assets, dtype=float)
for target in assets:
    for source in assets:
        x = wide_events[source].shift(1).fillna(0)
        y = wide_events[target]
        spillover_proxy.loc[target, source] = np.corrcoef(x, y)[0, 1] if x.std() > 0 and y.std() > 0 else np.nan
spillover_proxy


## 14. Model diagnostics: multivariate marked Hawkes–GPD

Key diagnostics for the final model:

1. **Stability:** spectral radius of the excitation matrix below one.
2. **Residual calibration:** time-rescaled residuals approximate independent exponential increments.
3. **Tail calibration:** GPD probability integral transform values are approximately uniform.
4. **Spillover interpretability:** economically meaningful direction and magnitude of off-diagonal excitation.
5. **Out-of-sample risk:** VaR exception and ES performance improve relative to simpler models.


In [ ]:
# 14.1 Final-model diagnostic placeholders
A_proxy = spillover_proxy.fillna(0).clip(lower=0).to_numpy()
spectral_radius_proxy = max(abs(np.linalg.eigvals(A_proxy))) if A_proxy.size else np.nan
final_diagnostics = {
    "spectral_radius_proxy": float(spectral_radius_proxy),
    "stable_proxy": bool(spectral_radius_proxy < 1) if pd.notna(spectral_radius_proxy) else None,
    "note": "Replace proxy diagnostics with MLE-based residual diagnostics in the final empirical implementation.",
}
final_diagnostics


## 15. Model comparison

The comparison table records model complexity, diagnostic status, and reason for escalation. In a completed study, add log-likelihood, AIC/BIC, out-of-sample VaR loss functions, ES scoring rules, and Diebold–Mariano tests where appropriate.


In [ ]:
# 15.1 Comparison table
model_comparison = pd.DataFrame([
    {"model": "Historical VaR/ES", "main_gain": "Transparent benchmark", "diagnostic_focus": "Exception rate and clustering", "escalate": ESCALATE_TO_POISSON_GPD},
    {"model": "Poisson-GPD", "main_gain": "Parametric exceedance frequency and severity", "diagnostic_focus": "Overdispersion and inter-arrival memory", "escalate": ESCALATE_TO_HAWKES_GPD},
    {"model": "Hawkes-GPD", "main_gain": "Self-exciting event times", "diagnostic_focus": "Residual clustering and mark predictability", "escalate": ESCALATE_TO_MARKED_HAWKES_GPD},
    {"model": "marked Hawkes-GPD", "main_gain": "Severity-dependent excitation", "diagnostic_focus": "Cross-asset dependence", "escalate": ESCALATE_TO_MULTIVARIATE_MARKED_HAWKES_GPD},
    {"model": "multivariate marked Hawkes-GPD", "main_gain": "Extreme-loss clustering and spillover", "diagnostic_focus": "Stability, residuals, spillover, forecasting", "escalate": False},
])
model_comparison


## 16. Risk forecasting

Forecasting should be conducted out-of-sample. The final risk engine should produce one-step-ahead and multi-step-ahead VaR/ES forecasts by combining conditional arrival intensity with GPD tail severity.

Suggested forecast outputs:

- Asset-level VaR/ES.
- Portfolio VaR/ES under fixed or time-varying weights.
- Probability of at least one extreme-loss event over a forecast horizon.
- Spillover stress scenarios based on high-mark events in source assets.


In [ ]:
# 16.1 Risk forecast scaffold
portfolio_weights = pd.Series(1 / len(assets), index=assets)
latest_losses = panel.pivot_table(index=DATE_COL, columns=ASSET_COL, values="loss").sort_index()
portfolio_loss = latest_losses.mul(portfolio_weights, axis=1).sum(axis=1)
portfolio_var, portfolio_es = historical_var_es(portfolio_loss.dropna(), TAIL_PROB)

risk_forecast = {
    "portfolio_historical_var": portfolio_var,
    "portfolio_historical_es": portfolio_es,
    "next_step_event_probability_proxy_by_asset": poisson_gpd_results.set_index(ASSET_COL)["lambda"].to_dict(),
}
risk_forecast


## 17. Conclusion

This template implements an academic modelling path for Vietnamese equity extreme losses. It begins with Historical VaR/ES, then escalates to Poisson–GPD, Hawkes–GPD, marked Hawkes–GPD, and finally multivariate marked Hawkes–GPD only when diagnostics motivate additional structure. The central empirical contribution is to distinguish unconditional tail risk from event clustering, mark-dependent excitation, and cross-asset spillover.


## 18. Appendix

### 18.1 Implementation checklist

- Replace the synthetic fallback with cleaned Vietnamese equity data.
- Select thresholds using mean residual life plots, parameter stability plots, and sensitivity analysis.
- Estimate Hawkes-family models by constrained maximum likelihood.
- Use robust standard errors or bootstrap intervals for excitation and spillover parameters.
- Run out-of-sample VaR/ES backtests.
- Report reproducibility metadata: data vintage, package versions, random seed, and hardware.

### 18.2 Suggested references to add in the manuscript

- Embrechts, Klüppelberg, and Mikosch on extreme value theory.
- Coles on statistical modelling of extremes.
- Hawkes on self-exciting point processes.
- Bacry, Mastromatteo, and Muzy on Hawkes processes in finance.
- McNeil, Frey, and Embrechts on quantitative risk management.


In [ ]:
# 18.3 Reproducibility metadata
import platform

metadata = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "random_seed": RANDOM_SEED,
    "data_path": str(DATA_PATH),
}
metadata
